# Chapter 7: Filtering Rows, Sorting Data, Ordering Values, and Working with Conditions

**Companion notebook** for *Beginner's Guide to Pandas* by Ravi Shankar

Run each cell in order. Exercises are at the end.

In [95]:
import pandas as pd
import numpy as np

# Filtering Rows, Sorting Data, Ordering Values, and Working with Conditions

## Introduction

Data manipulation is at the heart of data analysis. Whether you're cleaning a dataset, preparing it for visualization, or extracting specific subsets for investigation, knowing how to filter, sort, and apply conditions is essential. This chapter covers the practical techniques you'll use daily when working with pandas DataFrames and Series.

We'll start with the fundamentals of boolean filtering, build up to sorting and ordering, then explore the full toolkit of specialized filtering methods — from `isin()` and `between()` to `str.contains()` and `query()`. Along the way, you'll see how to combine these tools, handle missing values gracefully, and avoid the most common pitfalls.

---

## Filtering Rows with Boolean Conditions

### Basic Row Filtering

The most common way to filter rows is using **boolean indexing**. You create a condition that evaluates to `True` or `False` for each row, then use it to select matching rows.

In [96]:
import pandas as pd
import numpy as np

# Create sample dataset
sales = pd.DataFrame({
    'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Laptop'],
    'region': ['North', 'South', 'North', 'East', 'West'],
    'revenue': [1200, 45, 80, 350, 1150],
    'units_sold': [2, 15, 12, 5, 3]
})

# Filter for high-revenue items
high_revenue = sales[sales['revenue'] > 500]
print(high_revenue)

  product region  revenue  units_sold
0  Laptop  North     1200           2
4  Laptop   West     1150           3


Output:
```
  product region  revenue  units_sold
0  Laptop  North     1200           2
4  Laptop   West     1150           3
```

When you write `sales['revenue'] > 500`, pandas creates a **Boolean Series** — a column of `True`/`False` values, one for each row. Passing that Series inside `[]` keeps only the rows where the value is `True`.

In [97]:
# Inspect the boolean Series directly
price_check = sales['revenue'] > 500
print(price_check)
# 0     True
# 1    False
# 2    False
# 3    False
# 4     True
# dtype: bool

# Count how many rows match
print(price_check.sum())    # 2
print(price_check.mean())   # 0.4  (40% of rows match)

0     True
1    False
2    False
3    False
4     True
Name: revenue, dtype: bool
2
0.4


### Multiple Conditions

Combine conditions using logical operators: `&` (AND), `|` (OR), and `~` (NOT).

In [98]:
# Products in North region with revenue > 100
north_profitable = sales[(sales['region'] == 'North') & (sales['revenue'] > 100)]

# Products that are either Laptops OR have high units sold
popular = sales[(sales['product'] == 'Laptop') | (sales['units_sold'] > 10)]

# Everything except the North region
not_north = sales[~(sales['region'] == 'North')]

**Important:** Always use parentheses around each condition and the operators `&`, `|`, `~` instead of `and`, `or`, `not`.

Why? Python's `and`/`or` expect single boolean values, but pandas returns a Series of booleans — one per row. The `&`/`|`/`~` operators work element-wise on Series. Without parentheses, Python's operator precedence causes confusing errors:

In [99]:
# ❌ WRONG — raises ValueError
# result = sales[sales['revenue'] > 500 and sales['region'] == 'North']

# ❌ WRONG — operator precedence issue
# result = sales[sales['revenue'] > 500 & sales['units_sold'] > 2]

# ✅ CORRECT — each condition wrapped in parentheses
result = sales[(sales['revenue'] > 500) & (sales['region'] == 'North')]

### Using `.isin()` for Multiple Values

When checking if a column matches any value in a list, `.isin()` is cleaner and faster than chaining `|` operators:

In [100]:
# ❌ Verbose approach with OR operators
messy = sales[(sales['region'] == 'North') | (sales['region'] == 'East')]

# ✅ Clean approach with isin()
filtered = sales[sales['region'].isin(['North', 'East'])]

# Exclude certain products
no_peripherals = sales[~sales['product'].isin(['Mouse', 'Keyboard'])]

| Aspect | OR operators | `isin()` |
|--------|-------------|---------|
| Readability | Verbose | Clean |
| Maintainability | Hard to modify | Easy to edit |
| Performance | Slower with many values | Faster |
| Scalability | Becomes unwieldy | Scales well |

### Filtering for Missing Values

In [101]:
# Keep only rows with non-null values in a column
complete_data = sales[sales['revenue'].notna()]

# Keep only rows with null values
missing_data = sales[sales['revenue'].isna()]

# Drop rows with any missing values
clean_data = sales.dropna()

---

## Sorting Data

### Basic Sorting

Sort a DataFrame by one or more columns using `.sort_values()`:

In [102]:
# Sort by revenue in ascending order (default)
by_revenue = sales.sort_values(by='revenue')

# Sort by revenue in descending order
by_revenue_desc = sales.sort_values(by='revenue', ascending=False)

### Multi-Column Sorting

Sort by multiple columns to handle ties:

In [103]:
# Sort by region alphabetically, then by revenue from highest to lowest within each region
sorted_data = sales.sort_values(by=['region', 'revenue'], ascending=[True, False])

### Handling Missing Values During Sort

In [104]:
# Place NaN values at the beginning
sorted_na_first = sales.sort_values(by='revenue', na_position='first')

# Place NaN values at the end (default)
sorted_na_last = sales.sort_values(by='revenue', na_position='last')

### Sorting Series

Series objects have the same `.sort_values()` method:

In [105]:
# Create a Series
prices = pd.Series([1200, 45, 80, 350], index=['Laptop', 'Mouse', 'Keyboard', 'Monitor'])

# Sort by values
sorted_prices = prices.sort_values()

# Sort by index
sorted_by_index = prices.sort_index()

### Sorting by Index

In [106]:
# Sort DataFrame by index
sorted_by_index = sales.sort_index()

# Sort index in descending order
sorted_index_desc = sales.sort_index(ascending=False)

---

## Finding Extreme Values

Instead of sorting the entire dataset, use `.nlargest()` and `.nsmallest()` for better performance when you only need a few extreme values:

In [107]:
# Get top 3 products by revenue
top_3 = sales.nlargest(3, 'revenue')

# Get bottom 2 products by revenue
bottom_2 = sales.nsmallest(2, 'revenue')

# For Series
top_prices = prices.nlargest(2)

These methods are more efficient than sorting the full dataset when you only need a handful of results.

---

## Ordering Categorical Data

When working with categorical columns, you can control the sort order by defining a custom category ordering:

In [108]:
# Create a DataFrame with ordered categorical data
df = pd.DataFrame({
    'priority': pd.Categorical(['high', 'low', 'medium', 'high', 'low'],
                               categories=['low', 'medium', 'high'],
                               ordered=True),
    'task': ['A', 'B', 'C', 'D', 'E']
})

# Sort by the categorical order (low → medium → high)
sorted_by_priority = df.sort_values(by='priority')
print(sorted_by_priority)

  priority task
1      low    B
4      low    E
2   medium    C
0     high    A
3     high    D


Output:
```
  priority task
1      low    B
4      low    E
2   medium    C
0     high    A
3     high    D
```

---

## Conditional Assignment

### Using `.loc[]` with Conditions

Modify or create column values based on conditions using `.loc[]`:

In [109]:
# Create a new column based on conditions
sales['category'] = 'Standard'
sales.loc[sales['revenue'] > 1000, 'category'] = 'Premium'
sales.loc[sales['revenue'] < 100, 'category'] = 'Budget'

print(sales)

    product region  revenue  units_sold  category
0    Laptop  North     1200           2   Premium
1     Mouse  South       45          15    Budget
2  Keyboard  North       80          12    Budget
3   Monitor   East      350           5  Standard
4    Laptop   West     1150           3   Premium


Output:
```
    product region  revenue  units_sold  category
0    Laptop  North     1200           2   Premium
1     Mouse  South       45          15    Budget
2  Keyboard  North       80          12    Budget
3   Monitor   East      350           5  Standard
4    Laptop   West     1150           3   Premium
```

You can also combine multiple conditions:

In [110]:
df = sales.copy()

# Multiple conditions with AND
df.loc[(df['region'] == 'North') & (df['revenue'] > 500), 'bonus'] = 0.15
df.loc[(df['region'] != 'North') | (df['revenue'] <= 500), 'bonus'] = 0.10

### Using `.where()` and `.mask()`

Replace values conditionally without modifying the original structure:

In [111]:
# Keep values where condition is True, replace others with NaN
filtered_revenue = sales['revenue'].where(sales['revenue'] > 500)

# Inverse: replace where condition is True (with NaN)
masked_revenue = sales['revenue'].mask(sales['revenue'] <= 500)

# Replace with a specific value instead of NaN
capped_revenue = sales['revenue'].where(sales['revenue'] <= 1000, 1000)

---

## Combining Filtering and Sorting

Real-world analysis often requires both operations together:

In [112]:
# Get high-revenue products from North and East, sorted by revenue
result = sales[
    (sales['revenue'] > 100) &
    (sales['region'].isin(['North', 'East']))
].sort_values(by='revenue', ascending=False)

---

## Advanced Boolean Methods

Pandas provides several specialized methods for cleaner, more readable filtering.

### Quick Reference: Choosing Your Method

| Method | Best For | Returns |
|--------|----------|---------|
| `isin()` | Multiple specific values | Boolean Series |
| `between()` | Range filtering (inclusive) | Boolean Series |
| `str.contains()` | Partial string matches | Boolean Series |
| `query()` | Complex multi-condition filters | Filtered DataFrame |
| `.loc[]` | Conditional assignment | Modified DataFrame |

---

### `between()` for Range Filtering

`between()` checks if values fall within a range (inclusive by default), replacing verbose `>=` / `<=` combinations:

In [113]:
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank', 'Grace'],
    'age': [15, 22, 30, 40, 18, 35, 50]
})

# Find customers aged 18–35 (inclusive)
young_adults = df[df['age'].between(18, 35)]
print(young_adults)

      name  age
1      Bob   22
2  Charlie   30
4      Eve   18
5    Frank   35


Output:
```
      name  age
1      Bob   22
2  Charlie   30
4      Eve   18
5    Frank   35
```

**Inclusive vs. exclusive boundaries:**

In [114]:
# Inclusive (default): includes 18 and 35
inclusive = df[df['age'].between(18, 35)]

# Exclusive: excludes both endpoints
exclusive = df[df['age'].between(18, 35, inclusive='neither')]

# Left-inclusive: includes 18, excludes 35
left_only = df[df['age'].between(18, 35, inclusive='left')]

**Working with dates:**

In [115]:
df_dates = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=5),
    'sales': [100, 150, 200, 120, 180]
})

# Filter for a date range
q1_data = df_dates[df_dates['date'].between('2024-01-01', '2024-03-31')]

**NaN handling:** `NaN` values automatically return `False` in `between()`, so they are excluded from results.

---

### `str.contains()` for String Pattern Matching

`str.contains()` checks if each string in a column contains a specified pattern, returning a Boolean Series:

In [116]:
df = pd.DataFrame({
    'product': ['Widget A', 'Gadget X', 'Widget Pro', 'Tool Kit', 'Widget Lite'],
    'price': [10, 25, 15, 30, 8]
})

# Find all products containing "Widget"
widgets = df[df['product'].str.contains('Widget')]
print(widgets)

       product  price
0     Widget A     10
2   Widget Pro     15
4  Widget Lite      8


Output:
```
       product  price
0     Widget A     10
2   Widget Pro     15
4  Widget Lite      8
```

**Case-insensitive matching:**

In [117]:
df_emails = pd.DataFrame({
    'email': ['alice@gmail.com', 'bob@GMAIL.com', 'charlie@yahoo.com', 'diana@Gmail.com']
})

# Case-sensitive (default) — misses variations
case_sensitive = df_emails[df_emails['email'].str.contains('gmail')]

# Case-insensitive — catches all variations
case_insensitive = df_emails[df_emails['email'].str.contains('gmail', case=False)]

**Negation:**

In [118]:
# Products NOT containing "Widget"
non_widgets = df[~df['product'].str.contains('Widget', na=False)]

**Handling NaN values:** Without `na=False`, the operation raises an error if any cells contain `None` or `NaN`. Always specify `na=False` to treat missing values as non-matching:

In [119]:
df_with_nulls = pd.DataFrame({
    'product': ['Widget A', None, 'Gadget X', 'Widget Pro'],
    'price': [10, 15, 25, 20]
})

# ✅ Safe: na=False treats NaN as False (excluded)
result = df_with_nulls[df_with_nulls['product'].str.contains('Widget', na=False)]

**Regular expression patterns:**

In [120]:
df_emails = pd.DataFrame({
    'email': ['alice@gmail.com', 'bob@yahoo.co.uk', 'charlie@company.org', 'diana@test.net']
})

# Emails ending with .com
dot_com = df_emails[df_emails['email'].str.contains(r'\.com$', regex=True, na=False)]

# Valid email format
valid_emails = df_emails[
    df_emails['email'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', regex=True, na=False)
]

Common regex patterns:

| Pattern | Meaning |
|---------|---------|
| `^` | Start of string |
| `$` | End of string |
| `\d` | Any digit (0–9) |
| `\.` | Literal dot (escaped) |
| `\w` | Word character (letter, digit, underscore) |

---

### `query()` for Complex Multi-Condition Filters

`query()` evaluates a string expression to filter rows, offering SQL-like syntax that is often more readable than nested bracket notation:

In [121]:
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'age': [25, 35, 28, 42, 30],
    'salary': [50000, 75000, 60000, 90000, 65000]
})

# Bracket notation (nested)
bracket_result = df[(df['age'] > 25) & (df['salary'] > 60000)]

# query() — more readable
query_result = df.query('age > 25 and salary > 60000')

print(query_result)

    name  age  salary
1    Bob   35   75000
3  Diana   42   90000
4    Eve   30   65000


Output:
```
    name  age  salary
1    Bob   35   75000
3  Diana   42   90000
4    Eve   30   65000
```

**Key syntax rules inside `query()`:**
- Use `and`, `or`, `not` (not `&`, `|`, `~`)
- Reference columns by name directly (no `df['column']`)
- Wrap string values in quotes

**Using external variables with `@`:**

In [122]:
min_age = 30
target_dept = 'Sales'

# Use @ to reference Python variables inside query()
result = df.query('age >= @min_age or name == @target_dept')

Without `@`, pandas searches for a column named `min_age`. The `@` symbol tells pandas: "This is a variable from my code, not a column name."

**Column names with spaces:**

In [123]:
df_spaces = pd.DataFrame({
    'Product Name': ['Widget', 'Gadget', 'Tool'],
    'Unit Price': [10, 20, 15]
})

# Use backticks for column names with spaces
result = df_spaces.query('`Unit Price` > 15')

**When to use `query()`:**
- Three or more conditions
- You prefer SQL-like readability
- You need to reference external variables
- Column names contain spaces

---

## Combining Multiple Filtering Methods

Real-world filtering often requires mixing methods to handle different data types and conditions:

In [124]:
df = pd.DataFrame({
    'product_name': ['Laptop Pro', 'USB Cable', 'Monitor 4K', 'Keyboard RGB', 'Mouse Wireless'],
    'category': ['Electronics', 'Accessories', 'Electronics', 'Accessories', 'Accessories'],
    'price': [1200, 15, 400, 80, 35],
    'rating': [4.8, 4.2, 4.5, 4.7, 4.3],
    'in_stock': [True, True, False, True, True]
})

# High-quality accessories in stock within a price range, matching a name pattern
result = df[
    df['category'].isin(['Accessories']) &
    df['price'].between(20, 500) &
    (df['rating'] >= 4.5) &
    (df['in_stock'] == True) &
    df['product_name'].str.contains('RGB|Wireless', case=False, regex=True)
]

print(result)

   product_name     category  price  rating  in_stock
3  Keyboard RGB  Accessories     80     4.7      True


Output:
```
   product_name      category  price  rating  in_stock
3  Keyboard RGB  Accessories     80     4.7      True
```

---

## Handling Missing Values in Conditions

Be careful when your data contains `NaN` values — comparisons with NaN always return `False`:

In [125]:
df_with_nulls = pd.DataFrame({
    'product': ['Laptop', 'Mouse', 'Keyboard'],
    'price': [800, None, 75],
    'in_stock': [True, True, None]
})

# NaN comparisons return False
print(df_with_nulls['price'] > 50)
# 0     True
# 1    False   ← NaN > 50 is False
# 2     True

# Use notna() to filter only rows with valid prices
has_price = df_with_nulls['price'].notna()
expensive = df_with_nulls[has_price & (df_with_nulls['price'] > 50)]

# Find complete records (no NaN in any column)
complete_records = df_with_nulls.dropna()

# OR check each column explicitly
complete_records = df_with_nulls[
    df_with_nulls['price'].notna() &
    df_with_nulls['in_stock'].notna()
]

0     True
1    False
2     True
Name: price, dtype: bool


**How each method handles NaN:**

| Method | NaN behavior |
|--------|-------------|
| `isin()` | Returns `False` for NaN |
| `between()` | Returns `False` for NaN |
| `str.contains()` | Returns NaN (use `na=False`) |
| `isna()` / `notna()` | Explicitly detects NaN |

---

## Index Behavior After Filtering

Boolean indexing preserves the original index. This is often useful, but sometimes you want sequential numbering:

In [126]:
df = pd.DataFrame({
    'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor'],
    'price': [800, 25, 75, 300],
    'in_stock': [True, True, None, False]
}, index=['A', 'B', 'C', 'D'])

# Filter preserves original indices
expensive = df[df['price'] > 50]
print(expensive.index)  # Index(['A', 'C', 'D'], dtype='object')

# Reset to sequential integers
reset = expensive.reset_index(drop=True)
print(reset.index)  # RangeIndex(start=0, stop=3, step=1)

# Keep original index as a column
reset_with_col = expensive.reset_index()

Index(['A', 'C', 'D'], dtype='object')
RangeIndex(start=0, stop=3, step=1)


---

## Copy vs. View: Avoiding the SettingWithCopyWarning

Filtered DataFrames may be views of the original, not independent copies. To safely modify a filtered result, create an explicit copy:

In [127]:
df = pd.DataFrame({
    'product': ['Laptop', 'Mouse', 'Keyboard'],
    'price': [800, 25, 75], 
    'in_stock': [True, True, None]
})

# ✅ Safe: create an explicit copy before modifying
expensive_copy = df[df['price'] > 50].copy()
expensive_copy.loc[0, 'price'] = 900  # Does not affect original df

---

## Common Pitfalls and How to Fix Them

### ❌ Pitfall 1: Using `and`/`or` Instead of `&`/`|`

In [128]:
# ❌ WRONG
try:
    result = df[df['price'] > 50 and df['in_stock'] == True]
except ValueError as e:
    print(f"Error: {e}")

# ✅ CORRECT
result = df[(df['price'] > 50) & (df['in_stock'] == True)]

Error: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


### ❌ Pitfall 2: Forgetting Parentheses

In [129]:
# ❌ WRONG — operator precedence causes error
# result = df[df['price'] > 50 & df['in_stock'] == True]

# ✅ CORRECT
result = df[(df['price'] > 50) & (df['in_stock'] == True)]

### ❌ Pitfall 3: Type Mismatches

In [130]:
df = pd.DataFrame({'price': ['10', '20', '30'], 'quantity': [5, 3, 8]})

# ❌ WRONG — comparing string to number
try:
    result = df[df['price'] > 15]
except TypeError as e:
    print(f"Error: {e}")

# ✅ CORRECT — convert to numeric first
df['price'] = pd.to_numeric(df['price'])
result = df[df['price'] > 15]

Error: '>' not supported between instances of 'str' and 'int'


### ❌ Pitfall 4: Not Handling NaN in String Operations

In [131]:
df = pd.DataFrame({'product': ['Widget A', None, 'Widget Pro'], 'price': [10, 20, 30]})

# ❌ WRONG — may raise error or return unexpected NaN
# result = df[df['product'].str.contains('Widget')]

# ✅ CORRECT
result = df[df['product'].str.contains('Widget', na=False)]

### ❌ Pitfall 5: Forgetting `@` for Variables in `query()`

In [132]:
min_price = 15

# ❌ WRONG — pandas looks for a column named min_price
# result = df.query('price > min_price')

# ✅ CORRECT
result = df.query('price > @min_price')

---

## Decision Tree: Which Method to Use?

```
What do you need to filter?

├─ Single simple condition?
│  └─ Bracket notation: df[df['col'] > value]
│
├─ Multiple conditions (2–3)?
│  └─ Bracket notation: df[(df['col'] > 10) & (df['col2'] == 'A')]
│
├─ Multiple conditions (3+)?
│  └─ query(): df.query('col > 10 and col2 == "A"')
│
├─ Multiple values in one column?
│  └─ isin(): df[df['category'].isin(['A', 'B'])]
│
├─ Partial string match?
│  └─ str.contains(): df[df['product'].str.contains('Widget', na=False)]
│
├─ Range filtering?
│  └─ between(): df[df['price'].between(10, 20)]
│
├─ Need external variables?
│  └─ query() with @: df.query('price > @min_price')
│
└─ Assign values based on conditions?
   └─ .loc[]: df.loc[df['price'] > 100, 'tier'] = 'Premium'
```

---

## Practical Workflow Example

Here is a complete example combining multiple techniques:

In [133]:
# Dataset with customer purchases
purchases = pd.DataFrame({
    'customer_id': [1, 2, 1, 3, 2, 1, 3],
    'purchase_date': pd.date_range('2024-01-01', periods=7),
    'amount': [150, 75, 200, 50, 300, 120, 90],
    'status': ['completed', 'pending', 'completed', 'completed', 'completed',
               'completed', 'completed'],
    'region': ['East', 'West', 'East', 'North', 'West', 'East', 'North']
})

# Step 1: Filter for completed purchases over $100
significant = purchases[
    (purchases['status'] == 'completed') &
    (purchases['amount'] > 100)
]

# Step 2: Sort by date, most recent first
recent_significant = significant.sort_values(by='purchase_date', ascending=False)

# Step 3: Identify top spenders
top_customers = recent_significant.nlargest(3, 'amount')

# Step 4: Add a tier label
top_customers = top_customers.copy()
top_customers.loc[top_customers['amount'] >= 200, 'tier'] = 'Gold'
top_customers.loc[top_customers['amount'] < 200, 'tier'] = 'Silver'

print(top_customers)

   customer_id purchase_date  amount     status region    tier
4            2    2024-01-05     300  completed   West    Gold
2            1    2024-01-03     200  completed   East    Gold
0            1    2024-01-01     150  completed   East  Silver


---

## Performance Considerations

- **Combine conditions into a single filter** rather than chaining multiple filters — this avoids multiple passes over the data.
- **Use `.isin()` instead of multiple `|` conditions** for checking membership in large lists.
- **Use `.nlargest()` / `.nsmallest()`** instead of sorting the entire dataset when you only need top or bottom values.
- **Filter first, then sort** — sorting is expensive; reduce your data before sorting.
- **Use built-in aggregations** (`.sum()`, `.mean()`) rather than `.apply()` with custom functions when possible — built-ins are significantly faster on large datasets.

---

## Summary

| Operation | Method | Example |
|-----------|--------|---------|
| Filter rows | Boolean indexing | `df[df['col'] > 5]` |
| Multiple conditions | `&`, `\|`, `~` | `df[(df['col1'] > 5) & (df['col2'] == 'A')]` |
| Check membership | `.isin()` | `df[df['col'].isin([1, 2, 3])]` |
| Range filtering | `.between()` | `df[df['age'].between(18, 65)]` |
| Partial text match | `.str.contains()` | `df[df['email'].str.contains('@gmail', na=False)]` |
| Complex conditions | `.query()` | `df.query('age > 25 and salary > 50000')` |
| Sort ascending | `.sort_values()` | `df.sort_values(by='col')` |
| Sort descending | `.sort_values()` | `df.sort_values(by='col', ascending=False)` |
| Top/bottom values | `.nlargest()` / `.nsmallest()` | `df.nlargest(5, 'col')` |
| Conditional assignment | `.loc[]` | `df.loc[condition, 'col'] = value` |
| Replace conditionally | `.where()` / `.mask()` | `df['col'].where(condition)` |
| Check for nulls | `isna()` / `notna()` | `df[df['col'].notna()]` |

### Key Takeaways

- **Boolean indexing** with conditions is the primary filtering tool.
- Use **`&`, `|`, `~`** for combining conditions — never `and`, `or`, `not`.
- **Always wrap each condition in parentheses** to avoid operator precedence errors.
- **`.sort_values()`** handles single and multi-column sorting; **`.nlargest()` / `.nsmallest()`** are more efficient for finding extremes.
- **Categorical data** respects custom ordering during sorts.
- **`.loc[]`** enables conditional value assignment.
- Use **`na=False`** in `str.contains()` and check for nulls explicitly with `isna()` / `notna()`.
- Use **`@variable_name`** inside `query()` to reference Python variables.
- Create an explicit **`.copy()`** before modifying a filtered DataFrame to avoid the `SettingWithCopyWarning`.

---

## Practice Exercises

In [134]:
import pandas as pd
import numpy as np

# Create a sample dataset
df = pd.DataFrame({
    'employee': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'department': ['Sales', 'IT', 'Sales', 'HR', 'IT'],
    'salary': [50000, 75000, 55000, 60000, 80000],
    'years': [2, 5, 3, 1, 6],
    'bonus': [True, True, False, False, True]
})

print("Original DataFrame:")
print(df)

# Task 1: Find all IT employees
print("\n--- Task 1: IT employees ---")
print(df[df['department'] == 'IT'])

# Task 2: Find employees with salary between $55,000 and $75,000
print("\n--- Task 2: Salary between $55k–$75k ---")
print(df[df['salary'].between(55000, 75000)])

# Task 3: Create a 'seniority' column: 'Senior' if years >= 5, else 'Junior'
print("\n--- Task 3: Seniority column ---")
df['seniority'] = np.where(df['years'] >= 5, 'Senior', 'Junior')
print(df)

# Task 4: Find employees in Sales OR IT with bonus = True
print("\n--- Task 4: Sales/IT with bonus ---")
print(df[(df['department'].isin(['Sales', 'IT'])) & (df['bonus'] == True)])

# Task 5: Use query() to find IT employees earning more than $70,000
print("\n--- Task 5: IT earning >$70k using query() ---")
print(df.query('department == "IT" and salary > 70000'))

# Task 6: Sort employees by department, then by salary descending within each department
print("\n--- Task 6: Sort by department, then salary ---")
print(df.sort_values(by=['department', 'salary'], ascending=[True, False]))

Original DataFrame:
  employee department  salary  years  bonus
0    Alice      Sales   50000      2   True
1      Bob         IT   75000      5   True
2  Charlie      Sales   55000      3  False
3    Diana         HR   60000      1  False
4      Eve         IT   80000      6   True

--- Task 1: IT employees ---
  employee department  salary  years  bonus
1      Bob         IT   75000      5   True
4      Eve         IT   80000      6   True

--- Task 2: Salary between $55k–$75k ---
  employee department  salary  years  bonus
1      Bob         IT   75000      5   True
2  Charlie      Sales   55000      3  False
3    Diana         HR   60000      1  False

--- Task 3: Seniority column ---
  employee department  salary  years  bonus seniority
0    Alice      Sales   50000      2   True    Junior
1      Bob         IT   75000      5   True    Senior
2  Charlie      Sales   55000      3  False    Junior
3    Diana         HR   60000      1  False    Junior
4      Eve         IT   80000   

---

# Exercises

Test your understanding of this chapter's concepts.

### Exercise 1: Filter and Explore a Sales Dataset

Practice basic boolean filtering on a small sales DataFrame. Filter rows based on single and combined conditions to find specific subsets of sales data.

In [135]:
import pandas as pd

sales = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Widget', 'Doohickey', 'Gadget', 'Widget', 'Doohickey'],
    'region': ['North', 'South', 'South', 'North', 'North', 'East', 'East'],
    'units_sold': [120, 85, 200, 45, 310, 95, 60],
    'revenue': [2400, 1700, 4000, 900, 6200, 1900, 1200]
})

# TODO: Filter rows where units_sold is greater than 100
high_sales = None
print("High sales (units > 100):")
print(high_sales)

# TODO: Filter rows where region is 'North' AND revenue is greater than 1000
north_high_rev = None
print("\nNorth region with revenue > 1000:")
print(north_high_rev)

# TODO: Filter rows where product is either 'Widget' OR 'Gadget'
widget_or_gadget = None
print("\nWidgets or Gadgets:")
print(widget_or_gadget)

High sales (units > 100):
None

North region with revenue > 1000:
None

Widgets or Gadgets:
None


### Exercise 2: Sort a Student Grades DataFrame

Practice sorting a DataFrame containing student grades by single and multiple columns. Explore ascending and descending sort orders and find the top and bottom performers.

In [136]:
import pandas as pd

students = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'David', 'Eva', 'Frank'],
    'grade': [88, 72, 95, 72, 88, 61],
    'attendance': [95, 80, 98, 75, 90, 70],
    'subject': ['Math', 'Science', 'Math', 'Science', 'Science', 'Math']
})

# TODO: Sort the DataFrame by 'grade' in descending order
sorted_by_grade = None
print("Sorted by grade (descending):")
print(sorted_by_grade)

# TODO: Sort by 'grade' descending, then by 'attendance' descending
# (so ties in grade are broken by attendance)
sorted_multi = None
print("\nSorted by grade then attendance (both descending):")
print(sorted_multi)

# TODO: Find the top 2 students by grade using nlargest
top_2 = None
print("\nTop 2 students by grade:")
print(top_2)

Sorted by grade (descending):
None

Sorted by grade then attendance (both descending):
None

Top 2 students by grade:
None


### Exercise 3: Conditional Assignment on Employee Data

Use np.where and pd.cut to create new columns based on conditions in an employee DataFrame. Assign performance labels and salary bands based on existing column values.

In [137]:
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    'name': ['Sara', 'Tom', 'Uma', 'Victor', 'Wendy', 'Xander'],
    'department': ['HR', 'Engineering', 'Engineering', 'HR', 'Marketing', 'Marketing'],
    'salary': [52000, 95000, 110000, 48000, 67000, 73000],
    'performance_score': [78, 91, 85, 60, 88, 74]
})

# TODO: Use np.where to create a new column 'high_performer'
# It should be True if performance_score >= 80, otherwise False
employees['high_performer'] = None

# TODO: Use pd.cut to create a 'salary_band' column with three bins:
# 'Low' for salary < 60000, 'Mid' for 60000-90000, 'High' for > 90000
# Hint: use bins=[0, 60000, 90000, float('inf')] and labels=['Low', 'Mid', 'High']
employees['salary_band'] = None

print(employees[['name', 'performance_score', 'high_performer', 'salary', 'salary_band']])

     name  performance_score high_performer  salary salary_band
0    Sara                 78           None   52000        None
1     Tom                 91           None   95000        None
2     Uma                 85           None  110000        None
3  Victor                 60           None   48000        None
4   Wendy                 88           None   67000        None
5  Xander                 74           None   73000        None


### Exercise 4: Filter, Sort, and Handle Missing Values in Inventory Data

Work with a realistic inventory DataFrame that contains missing values. Filter out rows with missing data, apply multiple conditions, sort the results, and reset the index on the final filtered DataFrame.

In [138]:
import pandas as pd
import numpy as np

inventory = pd.DataFrame({
    'item': ['Bolt', 'Nut', 'Washer', 'Screw', 'Anchor', 'Pin', 'Clip'],
    'category': ['Fastener', 'Fastener', 'Fastener', 'Fastener', 'Heavy', 'Light', 'Light'],
    'stock': [500, np.nan, 1200, 300, np.nan, 800, 150],
    'unit_price': [0.05, 0.03, 0.02, 0.04, 2.50, 0.10, 0.08],
    'reorder_needed': [False, True, False, True, True, False, True]
})

# TODO: Filter out rows where 'stock' is NaN using notna()
clean_inventory = None
print("Inventory without missing stock:")
print(clean_inventory)

# TODO: From clean_inventory, filter rows where reorder_needed is True
# AND unit_price is less than 0.10
reorder_cheap = None
print("\nItems needing reorder with unit_price < 0.10:")
print(reorder_cheap)

# TODO: Sort clean_inventory by 'stock' ascending and reset the index
# (drop the old index so it doesn't become a column)
sorted_reset = None
print("\nClean inventory sorted by stock with reset index:")
print(sorted_reset)

Inventory without missing stock:
None

Items needing reorder with unit_price < 0.10:
None

Clean inventory sorted by stock with reset index:
None


---

# Solutions

*Scroll down only after you've attempted the exercises above.*

<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Filter and Explore a Sales Dataset

In [139]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Widget', 'Doohickey', 'Gadget', 'Widget', 'Doohickey'],
    'region': ['North', 'South', 'South', 'North', 'North', 'East', 'East'],
    'units_sold': [120, 85, 200, 45, 310, 95, 60],
    'revenue': [2400, 1700, 4000, 900, 6200, 1900, 1200]
})

# Filter rows where units_sold is greater than 100
high_sales = sales[sales['units_sold'] > 100]
print("High sales (units > 100):")
print(high_sales)

# Filter rows where region is 'North' AND revenue is greater than 1000
north_high_rev = sales[(sales['region'] == 'North') & (sales['revenue'] > 1000)]
print("\nNorth region with revenue > 1000:")
print(north_high_rev)

# Filter rows where product is either 'Widget' OR 'Gadget'
widget_or_gadget = sales[(sales['product'] == 'Widget') | (sales['product'] == 'Gadget')]
print("\nWidgets or Gadgets:")
print(widget_or_gadget)

High sales (units > 100):
  product region  units_sold  revenue
0  Widget  North         120     2400
2  Widget  South         200     4000
4  Gadget  North         310     6200

North region with revenue > 1000:
  product region  units_sold  revenue
0  Widget  North         120     2400
4  Gadget  North         310     6200

Widgets or Gadgets:
  product region  units_sold  revenue
0  Widget  North         120     2400
1  Gadget  South          85     1700
2  Widget  South         200     4000
4  Gadget  North         310     6200
5  Widget   East          95     1900


### Solution 2: Sort a Student Grades DataFrame

In [140]:
import pandas as pd
import numpy as np

students = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'David', 'Eva', 'Frank'],
    'grade': [88, 72, 95, 72, 88, 61],
    'attendance': [95, 80, 98, 75, 90, 70],
    'subject': ['Math', 'Science', 'Math', 'Science', 'Science', 'Math']
})

# Sort the DataFrame by 'grade' in descending order
sorted_by_grade = students.sort_values('grade', ascending=False)
print("Sorted by grade (descending):")
print(sorted_by_grade)

# Sort by 'grade' descending, then by 'attendance' descending
sorted_multi = students.sort_values(['grade', 'attendance'], ascending=[False, False])
print("\nSorted by grade then attendance (both descending):")
print(sorted_multi)

# Find the top 2 students by grade using nlargest
top_2 = students.nlargest(2, 'grade')
print("\nTop 2 students by grade:")
print(top_2)

Sorted by grade (descending):
    name  grade  attendance  subject
2  Carol     95          98     Math
0  Alice     88          95     Math
4    Eva     88          90  Science
1    Bob     72          80  Science
3  David     72          75  Science
5  Frank     61          70     Math

Sorted by grade then attendance (both descending):
    name  grade  attendance  subject
2  Carol     95          98     Math
0  Alice     88          95     Math
4    Eva     88          90  Science
1    Bob     72          80  Science
3  David     72          75  Science
5  Frank     61          70     Math

Top 2 students by grade:
    name  grade  attendance subject
2  Carol     95          98    Math
0  Alice     88          95    Math


### Solution 3: Conditional Assignment on Employee Data

In [141]:
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    'name': ['Sara', 'Tom', 'Uma', 'Victor', 'Wendy', 'Xander'],
    'department': ['HR', 'Engineering', 'Engineering', 'HR', 'Marketing', 'Marketing'],
    'salary': [52000, 95000, 110000, 48000, 67000, 73000],
    'performance_score': [78, 91, 85, 60, 88, 74]
})

# Use np.where to create a new column 'high_performer'
employees['high_performer'] = np.where(employees['performance_score'] >= 80, True, False)

# Use pd.cut to create a 'salary_band' column
employees['salary_band'] = pd.cut(
    employees['salary'],
    bins=[0, 60000, 90000, float('inf')],
    labels=['Low', 'Mid', 'High']
)

print(employees[['name', 'performance_score', 'high_performer', 'salary', 'salary_band']])

     name  performance_score  high_performer  salary salary_band
0    Sara                 78           False   52000         Low
1     Tom                 91            True   95000        High
2     Uma                 85            True  110000        High
3  Victor                 60           False   48000         Low
4   Wendy                 88            True   67000         Mid
5  Xander                 74           False   73000         Mid


### Solution 4: Filter, Sort, and Handle Missing Values in Inventory Data

In [142]:
import pandas as pd
import numpy as np

inventory = pd.DataFrame({
    'item': ['Bolt', 'Nut', 'Washer', 'Screw', 'Anchor', 'Pin', 'Clip'],
    'category': ['Fastener', 'Fastener', 'Fastener', 'Fastener', 'Heavy', 'Light', 'Light'],
    'stock': [500, np.nan, 1200, 300, np.nan, 800, 150],
    'unit_price': [0.05, 0.03, 0.02, 0.04, 2.50, 0.10, 0.08],
    'reorder_needed': [False, True, False, True, True, False, True]
})

# Filter out rows where 'stock' is NaN using notna()
clean_inventory = inventory[inventory['stock'].notna()]
print("Inventory without missing stock:")
print(clean_inventory)

# From clean_inventory, filter rows where reorder_needed is True
# AND unit_price is less than 0.10
reorder_cheap = clean_inventory[
    (clean_inventory['reorder_needed'] == True) &
    (clean_inventory['unit_price'] < 0.10)
]
print("\nItems needing reorder with unit_price < 0.10:")
print(reorder_cheap)

# Sort clean_inventory by 'stock' ascending and reset the index
sorted_reset = clean_inventory.sort_values('stock', ascending=True).reset_index(drop=True)
print("\nClean inventory sorted by stock with reset index:")
print(sorted_reset)

Inventory without missing stock:
     item  category   stock  unit_price  reorder_needed
0    Bolt  Fastener   500.0        0.05           False
2  Washer  Fastener  1200.0        0.02           False
3   Screw  Fastener   300.0        0.04            True
5     Pin     Light   800.0        0.10           False
6    Clip     Light   150.0        0.08            True

Items needing reorder with unit_price < 0.10:
    item  category  stock  unit_price  reorder_needed
3  Screw  Fastener  300.0        0.04            True
6   Clip     Light  150.0        0.08            True

Clean inventory sorted by stock with reset index:
     item  category   stock  unit_price  reorder_needed
0    Clip     Light   150.0        0.08            True
1   Screw  Fastener   300.0        0.04            True
2    Bolt  Fastener   500.0        0.05           False
3     Pin     Light   800.0        0.10           False
4  Washer  Fastener  1200.0        0.02           False
